# GANForge -- BigGAN (Standalone Notebook)

**Large-scale class-conditional high-quality generation**

This is a **self-contained extract** of the BigGAN section from
the [GANForge Master Notebook](GANForge_Master_Notebook.ipynb) -- every
shared setup utility it needs (imports, device selection, seeding,
`weights_init`, `show_images`, checkpointing, etc.) is included below in
Section 0, so this notebook runs **top to bottom on its own**, in Colab
or locally, without needing any other file.

> **⚠️ Compute note.** See the Section 0 GPU warning below. Heavier
> architectures (Progressive GAN, StyleGAN2, BigGAN, StackGAN, GameGAN)
> include a lightweight from-scratch training loop sized for a single
> GPU (or CPU, slowly) plus, where realistically available, code to load
> strong pretrained weights for inference.

<a id="0"></a>
## 0. Global Setup & Utilities

Everything shared across all twelve GANs lives here: installs, imports,
device selection, seeding, and small helper functions (`weights_init`,
`show_images`, `save_checkpoint`, `Logger`, an `EMA` helper, and a spectral
-norm-aware conv wrapper). Re-using these keeps every later section short
and focused on what makes *that* GAN different.

**🚀 Pro Tip.** In Google Colab: `Runtime -> Change runtime type -> T4 GPU`
before running anything. Everything here auto-detects CUDA and degrades
gracefully to CPU (slow but functional) if none is found.

In [ ]:
# Colab-only: uncomment to install packages not preinstalled in Colab.
# All of these are optional -- every dataset loader below has an offline
# synthetic fallback, so the notebook still runs without them.
# !pip install -q torchaudio datasets soundfile pillow-simd torchmetrics
# !pip install -q pytorch-fid clean-fid


In [ ]:
# ---------------------------------------------------------------------------
# Imports
# ---------------------------------------------------------------------------
import os
import io
import math
import time
import random
import warnings
from dataclasses import dataclass, field
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset

import torchvision
import torchvision.transforms as T
import torchvision.utils as vutils

import matplotlib.pyplot as plt

try:
    from tqdm.auto import tqdm
except ImportError:
    # Minimal shim so the notebook still runs if tqdm is somehow missing.
    def tqdm(iterable, *args, **kwargs):
        return iterable

warnings.filterwarnings("ignore")  # GAN training is noisy (deprecation spam); keep output readable

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)


In [ ]:
# ---------------------------------------------------------------------------
# Device setup
# ---------------------------------------------------------------------------
# We centralize device selection so every model/tensor in the notebook is
# created consistently. All training loops below check DEVICE.type == "cuda"
# to decide whether to use mixed precision (torch.cuda.amp), which roughly
# halves memory use and often speeds up training 2-3x on modern GPUs.
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"

if DEVICE.type == "cuda":
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️  No GPU found -- running on CPU. Reduce EPOCHS/BATCH_SIZE in "
          "every section for reasonable runtimes.")


In [ ]:
# ---------------------------------------------------------------------------
# Reproducibility
# ---------------------------------------------------------------------------
# GANs are notoriously sensitive to initialization -- fixing every RNG makes
# results reproducible across reruns, which matters a lot when you are
# debugging "why did my generator collapse this time".
SEED = 42

def set_seed(seed: int = SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)


In [ ]:
# ---------------------------------------------------------------------------
# Shared helper functions used across every GAN section below
# ---------------------------------------------------------------------------

def weights_init(m: nn.Module):
    '''DCGAN-style weight init (Radford et al., 2015).

    Why: GAN training is a saddle-point optimization between two networks
    that constantly move each other's loss landscape. Starting both nets
    from small, zero-centered weights (N(0, 0.02)) avoids one network
    dominating early purely due to init scale, which is a common cause of
    early, unrecoverable mode collapse.

    Uses isinstance (NOT a substring match on the class name) because many
    custom container classes in this notebook -- PGAN_ConvBlock,
    ModulatedConv2d, StyleConvBlock -- have "Conv" in their own name; a
    naive `"Conv" in classname` check would incorrectly try to initialize
    THOSE containers directly (they have no `.weight` of their own) instead
    of only their actual nn.Conv2d/nn.Linear/nn.BatchNorm2d leaf children.
    '''
    if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d, nn.Linear)):
        nn.init.normal_(m.weight.data, 0.0, 0.02)
        if getattr(m, "bias", None) is not None:
            nn.init.constant_(m.bias.data, 0)
    elif isinstance(m, (nn.BatchNorm2d, nn.InstanceNorm2d)) and getattr(m, "affine", False):
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)


def show_images(images: torch.Tensor, title: str = "", nrow: int = 8, figsize=(8, 8), normalize=True):
    '''Display a batch of image tensors (B, C, H, W) in [-1, 1] or [0, 1] as a grid.'''
    images = images.detach().cpu()
    grid = vutils.make_grid(images, nrow=nrow, normalize=normalize, value_range=(-1, 1) if normalize else None, padding=2)
    plt.figure(figsize=figsize)
    plt.axis("off")
    plt.title(title)
    npimg = grid.numpy()
    if npimg.shape[0] == 1:
        plt.imshow(npimg[0], cmap="gray")
    else:
        plt.imshow(np.transpose(npimg, (1, 2, 0)))
    plt.show()


def save_checkpoint(state: dict, path: str):
    '''Persist model/optimizer state so long training runs survive a Colab disconnect.'''
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    torch.save(state, path)


def load_checkpoint(path: str, map_location=None):
    if not os.path.exists(path):
        return None
    return torch.load(path, map_location=map_location or DEVICE)


class EMA:
    '''Exponential Moving Average of generator weights.

    Why: instantaneous generator weights oscillate a lot during adversarial
    training (they are chasing a moving discriminator). Averaging weights
    over time (as StyleGAN2 and BigGAN both do) gives a much smoother,
    higher-quality generator for *inference*, at zero extra training cost.
    '''
    def __init__(self, model: nn.Module, decay: float = 0.999):
        self.decay = decay
        self.shadow = {k: v.detach().clone() for k, v in model.state_dict().items()}

    @torch.no_grad()
    def update(self, model: nn.Module):
        for k, v in model.state_dict().items():
            if v.dtype.is_floating_point:
                self.shadow[k].mul_(self.decay).add_(v.detach(), alpha=1 - self.decay)
            else:
                self.shadow[k] = v.detach().clone()

    def copy_to(self, model: nn.Module):
        model.load_state_dict(self.shadow, strict=True)


class AvgMeter:
    '''Running mean tracker for loss logging.'''
    def __init__(self):
        self.sum, self.n = 0.0, 0

    def update(self, val, k=1):
        self.sum += float(val) * k
        self.n += k

    @property
    def avg(self):
        return self.sum / max(self.n, 1)


def denorm(x: torch.Tensor) -> torch.Tensor:
    '''Map a tensor from [-1, 1] (tanh output range) back to [0, 1] for display/saving.'''
    return (x.clamp(-1, 1) + 1) / 2


CHECKPOINT_DIR = "checkpoints"
SAMPLES_DIR = "samples"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(SAMPLES_DIR, exist_ok=True)
print("Shared utilities ready.")


<a id="6"></a>
## 6. BigGAN -- Large-scale class-conditional high-quality generation

### 6.1 Problem Statement & Why this GAN

Brock et al. (2018) asked: what happens if you take class-conditional
GANs (Section 2/3) and simply **scale everything up** -- batch size (up
to 2048), model width, and dataset diversity (all 1000 ImageNet classes) --
while fixing the specific instabilities that scale exposes? Their answer
combines **self-attention** (global, not just local receptive fields),
**spectral normalization** on every weight (Lipschitz-constrained layers
for stability), a **class-conditional projection discriminator**, and a
**hierarchical latent** (different chunks of `z` fed to different
resolutions). This is the **large-scale, high-fidelity class-conditional
generation** problem -- more classes, more diversity, sharper detail than
cGAN/AC-GAN can achieve.

We implement every one of these architectural ideas at CIFAR-10 scale
(all 10 classes, 32x32) -- full BigGAN needs ImageNet at 128-512px and
enormous batch sizes to show its real advantage, which is why we also
provide official pretrained-weight loading below.

### 6.2 Data Collection & Preparation

In [ ]:
BIGGAN_IMG_SIZE = 32
BIGGAN_BATCH_SIZE = 64
BIGGAN_NZ = 120                 # divisible by BIGGAN_N_CHUNKS below (hierarchical latent)
BIGGAN_N_CLASSES = 10
BIGGAN_EPOCHS = 8
BIGGAN_CLASS_NAMES = ["airplane", "automobile", "bird", "cat", "deer", "dog", "frog", "horse", "ship", "truck"]

biggan_transform = T.Compose([T.ToTensor(), T.Normalize((0.5,) * 3, (0.5,) * 3)])

try:
    biggan_dataset = torchvision.datasets.CIFAR10(root="data", train=True, download=True, transform=biggan_transform)
    print(f"Loaded real CIFAR-10 (all classes): {len(biggan_dataset)} images")
except Exception as e:
    print(f"CIFAR-10 download failed ({e}); falling back to synthetic per-class textures.")
    rng = np.random.default_rng(SEED)
    imgs, labels = [], []
    for c in range(BIGGAN_N_CLASSES):
        base_color = torch.tensor(rng.uniform(-1, 1, 3), dtype=torch.float32)   # dtype pinned: rng.uniform is float64, and float32+float64 silently promotes to float64 (breaks Conv2d's float32 weights downstream)
        for _ in range(300):
            img = (torch.randn(3, BIGGAN_IMG_SIZE, BIGGAN_IMG_SIZE) * 0.1 + base_color.view(3, 1, 1)).clamp(-1, 1)
            imgs.append(img); labels.append(c)
    biggan_dataset = TensorDataset(torch.stack(imgs), torch.tensor(labels))

biggan_loader = DataLoader(biggan_dataset, batch_size=BIGGAN_BATCH_SIZE, shuffle=True, drop_last=True)
real_batch, real_labels = next(iter(biggan_loader))
show_images(real_batch[:32], title="BigGAN training data (10 classes)", nrow=8)


### 6.3 Model Architecture

In [ ]:
def sn_conv2d(*args, **kwargs):
    '''Spectral normalization rescales a layer's weight matrix by (an
    estimate of) its largest singular value every forward pass, so the
    layer is provably 1-Lipschitz. BigGAN applies this to EVERY weight in
    BOTH networks (not just D, as in earlier spectral-norm GANs) -- the
    paper found this is what lets you push batch size/width up without
    training collapsing.'''
    return nn.utils.spectral_norm(nn.Conv2d(*args, **kwargs))


def sn_linear(*args, **kwargs):
    return nn.utils.spectral_norm(nn.Linear(*args, **kwargs))


class SelfAttention(nn.Module):
    '''Non-local self-attention (SAGAN, used inside BigGAN). Convolutions
    only ever see a local receptive field; attention lets every spatial
    location attend to every other one, which matters for images with
    long-range structure (e.g. a dog's two ears must match, even though
    they may be far apart at low resolution).'''
    def __init__(self, channels):
        super().__init__()
        self.query = sn_conv2d(channels, max(1, channels // 8), 1)
        self.key = sn_conv2d(channels, max(1, channels // 8), 1)
        self.value = sn_conv2d(channels, channels, 1)
        self.gamma = nn.Parameter(torch.zeros(1))   # starts at 0 -> block starts as identity, attention "phased in" as gamma learns to grow

    def forward(self, x):
        b, c, h, w = x.shape
        q = self.query(x).view(b, -1, h * w).permute(0, 2, 1)
        k = self.key(x).view(b, -1, h * w)
        attn = torch.softmax(torch.bmm(q, k), dim=-1)
        v = self.value(x).view(b, -1, h * w)
        out = torch.bmm(v, attn.permute(0, 2, 1)).view(b, c, h, w)
        return x + self.gamma * out


class ConditionalBatchNorm2d(nn.Module):
    '''BatchNorm whose per-channel gain/bias are PREDICTED from the
    conditioning vector (class embedding concatenated with a z-chunk) via
    small linear layers, instead of being plain learned parameters. This
    is how the class label (and, uniquely to BigGAN, part of the noise
    vector too) is injected at every single resolution of the generator.'''
    def __init__(self, num_features, cond_dim):
        super().__init__()
        self.num_features = num_features
        self.bn = nn.BatchNorm2d(num_features, affine=False)
        self.gain = sn_linear(cond_dim, num_features)
        self.bias = sn_linear(cond_dim, num_features)

    def forward(self, x, cond):
        out = self.bn(x)
        gain = (1 + self.gain(cond)).view(-1, self.num_features, 1, 1)
        bias = self.bias(cond).view(-1, self.num_features, 1, 1)
        return out * gain + bias


class GResBlockUp(nn.Module):
    '''Residual upsampling block: BN-ReLU-Upsample-Conv twice, with a
    matching upsample+1x1-conv skip path (pre-activation ResNet style).'''
    def __init__(self, in_c, out_c, cond_dim):
        super().__init__()
        self.bn1 = ConditionalBatchNorm2d(in_c, cond_dim)
        self.conv1 = sn_conv2d(in_c, out_c, 3, 1, 1)
        self.bn2 = ConditionalBatchNorm2d(out_c, cond_dim)
        self.conv2 = sn_conv2d(out_c, out_c, 3, 1, 1)
        self.skip_conv = sn_conv2d(in_c, out_c, 1, bias=False)
        self.act = nn.ReLU(True)

    def forward(self, x, cond):
        h = self.act(self.bn1(x, cond))
        h = F.interpolate(h, scale_factor=2, mode="nearest")
        h = self.conv1(h)
        h = self.act(self.bn2(h, cond))
        h = self.conv2(h)
        skip = self.skip_conv(F.interpolate(x, scale_factor=2, mode="nearest"))
        return h + skip


class BigGAN_Generator(nn.Module):
    '''Hierarchical latent: z is split into N_CHUNKS pieces, one consumed by
    the initial projection and one by EACH residual block's conditional
    BatchNorm (concatenated with the shared class embedding). This gives
    the generator direct, per-resolution control over the noise -- BigGAN's
    ablations show this measurably improves sample quality over feeding
    the same z everywhere.'''
    N_CHUNKS = 4

    def __init__(self, nz=BIGGAN_NZ, n_classes=BIGGAN_N_CLASSES, embed_dim=128, base_ch=96, bottom=4):
        super().__init__()
        assert nz % self.N_CHUNKS == 0
        self.chunk_size = nz // self.N_CHUNKS
        self.base_ch, self.bottom = base_ch, bottom
        self.class_embed = nn.Embedding(n_classes, embed_dim)
        self.fc = sn_linear(self.chunk_size, base_ch * 8 * bottom * bottom)
        cond_dim = embed_dim + self.chunk_size
        self.block1 = GResBlockUp(base_ch * 8, base_ch * 4, cond_dim)   # 4 -> 8
        self.block2 = GResBlockUp(base_ch * 4, base_ch * 2, cond_dim)   # 8 -> 16
        self.attn = SelfAttention(base_ch * 2)
        self.block3 = GResBlockUp(base_ch * 2, base_ch, cond_dim)       # 16 -> 32
        self.final_bn = nn.BatchNorm2d(base_ch)
        self.final_conv = sn_conv2d(base_ch, 3, 3, 1, 1)

    def forward(self, z, y):
        chunks = z.split(self.chunk_size, dim=1)
        y_embed = self.class_embed(y)
        x = self.fc(chunks[0]).view(-1, self.base_ch * 8, self.bottom, self.bottom)
        x = self.block1(x, torch.cat([chunks[1], y_embed], dim=1))
        x = self.block2(x, torch.cat([chunks[2], y_embed], dim=1))
        x = self.attn(x)
        x = self.block3(x, torch.cat([chunks[3], y_embed], dim=1))
        x = F.relu(self.final_bn(x))
        return torch.tanh(self.final_conv(x))


class DResBlockDown(nn.Module):
    '''`activate_first=False` matches BigGAN/SNGAN's "OptimizedBlock": the
    very FIRST discriminator block skips its leading activation, since
    there are no learned features yet to activate -- applying ReLU
    directly to the raw image would needlessly clip away every negative
    pixel value (our data is normalized to [-1, 1], so that's half of it).

    Note the activation is explicitly `inplace=False`: an in-place ReLU
    applied to this block's input `x` would, when `x` IS the generator's
    Tanh output (as happens when the generator is updated), silently
    corrupt the exact tensor autograd needs to compute Tanh's local
    gradient during `loss_g.backward()` -- PyTorch detects this and raises
    a "modified by an inplace operation" RuntimeError.'''
    def __init__(self, in_c, out_c, downsample=True, activate_first=True):
        super().__init__()
        self.conv1 = sn_conv2d(in_c, out_c, 3, 1, 1)
        self.conv2 = sn_conv2d(out_c, out_c, 3, 1, 1)
        self.skip_conv = sn_conv2d(in_c, out_c, 1, bias=False)
        self.downsample = downsample
        self.activate_first = activate_first
        self.act = nn.ReLU(inplace=False)

    def forward(self, x):
        h = self.act(x) if self.activate_first else x
        h = self.conv1(h)
        h = self.conv2(self.act(h))
        skip = self.skip_conv(x)
        if self.downsample:
            h, skip = F.avg_pool2d(h, 2), F.avg_pool2d(skip, 2)
        return h + skip


class BigGAN_Discriminator(nn.Module):
    '''Projection discriminator (Miyato & Koyama, 2018): instead of
    concatenating the label as an extra channel (cGAN-style), the class
    embedding is combined with the image FEATURES via an inner product
    added to the unconditional real/fake score. This is a much stronger,
    more scalable conditioning signal for discriminators with many classes.'''
    def __init__(self, n_classes=BIGGAN_N_CLASSES, base_ch=96):
        super().__init__()
        self.block1 = DResBlockDown(3, base_ch, activate_first=False)   # 32 -> 16 (no leading activation -- see DResBlockDown docstring)
        self.attn = SelfAttention(base_ch)
        self.block2 = DResBlockDown(base_ch, base_ch * 2)     # 16 -> 8
        self.block3 = DResBlockDown(base_ch * 2, base_ch * 4)  # 8 -> 4
        self.block4 = DResBlockDown(base_ch * 4, base_ch * 8, downsample=False)  # 4 -> 4
        self.act = nn.ReLU(inplace=False)
        self.fc = sn_linear(base_ch * 8, 1)
        self.class_embed = nn.Embedding(n_classes, base_ch * 8)

    def forward(self, x, y):
        h = self.block1(x)
        h = self.attn(h)
        h = self.block2(h)
        h = self.block3(h)
        h = self.block4(h)
        h = self.act(h).sum(dim=[2, 3])          # global sum pooling (BigGAN uses sum, not mean, pooling)
        out = self.fc(h).view(-1)
        proj = (self.class_embed(y) * h).sum(dim=1)   # projection term
        return out + proj


G_biggan = BigGAN_Generator().to(DEVICE)
D_biggan = BigGAN_Discriminator().to(DEVICE)
print(f"BigGAN-lite: {sum(p.numel() for p in G_biggan.parameters())/1e6:.1f}M generator params, "
      f"{sum(p.numel() for p in D_biggan.parameters())/1e6:.1f}M discriminator params")


### 6.4 Loss Functions & Optimizers

In [ ]:
# Hinge loss (Lim & Ye / Tran et al.), the loss BigGAN actually uses: it
# only penalizes D while its margin is violated (D(real) < 1 or D(fake) >
# -1), which pairs well with spectral normalization's Lipschitz bound and
# empirically gives cleaner gradients than BCE at this scale.
def biggan_d_hinge_loss(logits_real, logits_fake):
    return F.relu(1.0 - logits_real).mean() + F.relu(1.0 + logits_fake).mean()

def biggan_g_hinge_loss(logits_fake):
    return -logits_fake.mean()

# BigGAN uses a LOWER learning rate + more D steps per G step than most
# GANs in this notebook, and, crucially, DIFFERENT learning rates for G/D
# (Two-Timescale Update Rule, TTUR) -- D is allowed to adapt faster.
biggan_opt_G = torch.optim.Adam(G_biggan.parameters(), lr=1e-4, betas=(0.0, 0.999))
biggan_opt_D = torch.optim.Adam(D_biggan.parameters(), lr=4e-4, betas=(0.0, 0.999))
BIGGAN_D_STEPS_PER_G_STEP = 2

biggan_ema = EMA(G_biggan, decay=0.999)  # BigGAN inference always uses the EMA generator, never raw weights


### 6.5 Training Loop

In [ ]:
# ---------------------------------------------------------------------------
# 6.5a Lightweight from-scratch training
# ---------------------------------------------------------------------------
biggan_fixed_z = torch.randn(BIGGAN_N_CLASSES * 4, BIGGAN_NZ, device=DEVICE)
biggan_fixed_y = torch.arange(BIGGAN_N_CLASSES, device=DEVICE).repeat_interleave(4)
biggan_history = []

for epoch in range(BIGGAN_EPOCHS):
    meters = {"G": AvgMeter(), "D": AvgMeter()}
    pbar = tqdm(biggan_loader, desc=f"[BigGAN] epoch {epoch+1}/{BIGGAN_EPOCHS}")
    d_step_counter = 0
    for real, y in pbar:
        real, y = real.to(DEVICE), y.to(DEVICE)
        b = real.size(0)

        # ---- Discriminator (D_STEPS_PER_G_STEP updates per G update) ----
        D_biggan.zero_grad(set_to_none=True)
        z = torch.randn(b, BIGGAN_NZ, device=DEVICE)
        fake_y = torch.randint(0, BIGGAN_N_CLASSES, (b,), device=DEVICE)
        with torch.no_grad():
            fake = G_biggan(z, fake_y)
        logits_real = D_biggan(real, y)
        logits_fake = D_biggan(fake, fake_y)
        loss_d = biggan_d_hinge_loss(logits_real, logits_fake)
        loss_d.backward()
        biggan_opt_D.step()
        d_step_counter += 1

        if d_step_counter % BIGGAN_D_STEPS_PER_G_STEP == 0:
            # ---- Generator ----
            G_biggan.zero_grad(set_to_none=True)
            z = torch.randn(b, BIGGAN_NZ, device=DEVICE)
            fake_y = torch.randint(0, BIGGAN_N_CLASSES, (b,), device=DEVICE)
            fake = G_biggan(z, fake_y)
            logits_fake_g = D_biggan(fake, fake_y)
            loss_g = biggan_g_hinge_loss(logits_fake_g)
            loss_g.backward()
            biggan_opt_G.step()
            biggan_ema.update(G_biggan)
            meters["G"].update(loss_g.item())

        meters["D"].update(loss_d.item())
        pbar.set_postfix(loss_D=f"{loss_d.item():.3f}", loss_G=f"{meters['G'].avg:.3f}")

    biggan_history.append((meters["G"].avg, meters["D"].avg))
    with torch.no_grad():
        eval_G = BigGAN_Generator().to(DEVICE)
        biggan_ema.copy_to(eval_G)
        eval_G.eval()
        sample = eval_G(biggan_fixed_z, biggan_fixed_y)
    show_images(sample, title=f"BigGAN-lite (EMA) samples after epoch {epoch+1}", nrow=BIGGAN_N_CLASSES)
    save_checkpoint({"G": G_biggan.state_dict(), "D": D_biggan.state_dict(), "ema": biggan_ema.shadow},
                     f"{CHECKPOINT_DIR}/biggan_epoch{epoch+1}.pt")

print("BigGAN-lite training complete.")


In [ ]:
# ---------------------------------------------------------------------------
# 6.5b Loading strong pretrained weights (inference-only mode)
# ---------------------------------------------------------------------------
# `pytorch_pretrained_biggan` (Hugging Face) wraps official BigGAN-deep
# checkpoints trained on all 1000 ImageNet classes at 128/256/512px --
# genuinely large-scale, unlike our CIFAR-10-scale model above.
def try_load_pretrained_biggan(model_name="biggan-deep-128"):
    try:
        from pytorch_pretrained_biggan import BigGAN
        model = BigGAN.from_pretrained(model_name).to(DEVICE).eval()
        print(f"Loaded official pretrained {model_name} (1000 ImageNet classes).")
        return model
    except Exception as e:
        print(f"Pretrained BigGAN unavailable ({e}). Install with "
              f"'pip install pytorch-pretrained-biggan' and ensure network access. "
              f"Falling back to the from-scratch BigGAN-lite model trained above.")
        return None

biggan_pretrained = try_load_pretrained_biggan()


### 6.6 Inference / Generation Function

In [ ]:
@torch.no_grad()
def biggan_generate(class_id: int, n_samples: int = 8, truncation: float = 0.5, seed: int = None, use_pretrained: bool = False) -> torch.Tensor:
    '''BigGAN's truncation trick: sample z from a NORMAL, then resample any
    component with |z| > truncation (truncated normal) -- lower truncation
    means safer/sharper/less diverse samples, exactly analogous to
    StyleGAN2's w-space truncation but applied directly to z.'''
    if seed is not None:
        torch.manual_seed(seed)
    if use_pretrained and biggan_pretrained is not None:
        from pytorch_pretrained_biggan import truncated_noise_sample, one_hot_from_int
        noise = torch.from_numpy(truncated_noise_sample(truncation=truncation, batch_size=n_samples)).to(DEVICE)
        one_hot = torch.from_numpy(one_hot_from_int([class_id] * n_samples, batch_size=n_samples)).to(DEVICE)
        return biggan_pretrained(noise, one_hot, truncation)

    eval_G = BigGAN_Generator().to(DEVICE)
    biggan_ema.copy_to(eval_G)
    eval_G.eval()
    z = torch.randn(n_samples, BIGGAN_NZ, device=DEVICE).clamp(-truncation * 2, truncation * 2)
    y = torch.full((n_samples,), class_id, dtype=torch.long, device=DEVICE)
    return eval_G(z, y)

show_images(biggan_generate(class_id=3, n_samples=8), title=f"BigGAN inference: class='{BIGGAN_CLASS_NAMES[3]}'", nrow=8, figsize=(10, 2))


### 6.7 Visualization & Results

In [ ]:
gs, ds = zip(*biggan_history)
plt.figure(figsize=(6, 4)); plt.plot(gs, label="G"); plt.plot(ds, label="D")
plt.legend(); plt.title("BigGAN-lite training losses"); plt.show()

# All 10 classes, EMA weights -- the core deliverable of large-scale
# class-conditional generation.
eval_G = BigGAN_Generator().to(DEVICE)
biggan_ema.copy_to(eval_G)
eval_G.eval()
with torch.no_grad():
    z = torch.randn(BIGGAN_N_CLASSES * 4, BIGGAN_NZ, device=DEVICE)
    y = torch.arange(BIGGAN_N_CLASSES, device=DEVICE).repeat_interleave(4)
    grid = eval_G(z, y)
show_images(grid, title="One row per class: " + ", ".join(BIGGAN_CLASS_NAMES), nrow=4, figsize=(6, 15))

# Truncation sweep for a single class.
with torch.no_grad():
    z_fixed = torch.randn(8, BIGGAN_NZ, device=DEVICE)
    y_fixed = torch.full((8,), 7, dtype=torch.long, device=DEVICE)  # "horse"
    for trunc in [0.3, 0.7, 1.2]:
        imgs = eval_G(z_fixed.clamp(-trunc * 2, trunc * 2), y_fixed)
        show_images(imgs, title=f"Truncation={trunc}, class='horse'", nrow=8, figsize=(12, 2))


### 6.8 User Input → Output Demo

In [ ]:
USER_REQUESTED_CLASS = "ship"
TRUNCATION = 0.5
N_SAMPLES_TO_GENERATE = 8

class_idx = BIGGAN_CLASS_NAMES.index(USER_REQUESTED_CLASS)
demo = biggan_generate(class_idx, n_samples=N_SAMPLES_TO_GENERATE, truncation=TRUNCATION, seed=11)
show_images(demo, title=f"User requested: '{USER_REQUESTED_CLASS}', truncation={TRUNCATION}", nrow=4)


### 6.9 Key Insights & Limitations of this GAN

- 💡 **Insight.** Almost none of BigGAN's individual components are new
  (self-attention: SAGAN; spectral norm: Miyato et al.; projection
  discriminator: Miyato & Koyama; hinge loss: geometric GAN) -- its
  contribution was rigorously combining them AND showing they keep working
  as batch size and model width scale up by 10-50x, which is a genuinely
  hard empirical/engineering result, not just an architecture.
- 🚀 **Pro Tip.** Always evaluate/deploy the **EMA** generator, never the
  raw training weights -- the gap in sample quality is large and free.
- ⚠️ **Warning.** BigGAN is notoriously prone to **late-training collapse**
  even after many stable epochs, which is why the original paper
  checkpoints frequently and picks the best checkpoint by FID rather than
  training to a fixed final step.

**This GAN solved the problem of: large-scale, high-fidelity,
class-conditional generation across many diverse classes -- pushing
conditional-GAN sample quality far beyond what cGAN/AC-GAN achieve by
combining attention, spectral normalization, and a stronger conditioning
mechanism at scale.**